# 📊 Prometheus + Grafana Monitoring

**Role:** Installs and runs a system monitoring stack inside this notebook.
**Focus:** CPU, Memory, Disk, and Network metrics.

**Components:**
* **Prometheus:** Time-series database (Port 9090)
* **Grafana:** Dashboard UI (Port 3000)
* **Node Exporter:** System metrics agent (Port 9100)

## 1. Installation (Run Once)
This cell downloads the standalone binaries. We use a robust script that checks if files exist to prevent errors.

In [ ]:
%%bash

# Define Versions
PROM_VER="2.45.0"
GRAFANA_VER="10.0.3"
NODE_EXP_VER="1.6.1"

# Create directories if they don't exist
mkdir -p prometheus_bin grafana_bin

# --- Download Prometheus ---
if [ ! -f prometheus_bin/prometheus ]; then
    echo "⬇️  Downloading Prometheus..."
    wget -q -nc https://github.com/prometheus/prometheus/releases/download/v${PROM_VER}/prometheus-${PROM_VER}.linux-amd64.tar.gz
    tar xf prometheus-${PROM_VER}.linux-amd64.tar.gz --strip-components=1 -C prometheus_bin
    echo "✅ Prometheus Installed"
else
    echo "⏩ Prometheus already installed."
fi

# --- Download Grafana ---
if [ ! -f grafana_bin/bin/grafana-server ]; then
    echo "⬇️  Downloading Grafana..."
    wget -q -nc https://dl.grafana.com/oss/release/grafana-${GRAFANA_VER}.linux-amd64.tar.gz
    tar xf grafana-${GRAFANA_VER}.linux-amd64.tar.gz --strip-components=1 -C grafana_bin
    echo "✅ Grafana Installed"
else
    echo "⏩ Grafana already installed."
fi

# --- Download Node Exporter ---
if [ ! -f prometheus_bin/node_exporter ]; then
    echo "⬇️  Downloading Node Exporter..."
    wget -q -nc https://github.com/prometheus/node_exporter/releases/download/v${NODE_EXP_VER}/node_exporter-${NODE_EXP_VER}.linux-amd64.tar.gz
    tar xf node_exporter-${NODE_EXP_VER}.linux-amd64.tar.gz --strip-components=1 -C prometheus_bin
    echo "✅ Node Exporter Installed"
else
    echo "⏩ Node Exporter already installed."
fi

# --- Cleanup ---
rm *.tar.gz 2>/dev/null || true
echo "🎉 All binaries ready!"

## 2. Generate Configuration
We generate a `prometheus.yml` that configures Prometheus to scrape the Node Exporter every 5 seconds.

In [ ]:
prometheus_config = """
global:
  scrape_interval: 5s

scrape_configs:
  - job_name: 'jupyter_cpu_node'
    static_configs:
      - targets: ['localhost:9100']  # Node Exporter
"""

with open("prometheus.yml", "w") as f:
    f.write(prometheus_config)
print("✅ 'prometheus.yml' configuration created.")

## 3. Start Services
Launch the background processes. Logs are redirected to `.log` files in your directory.

In [ ]:
import subprocess
import os

BASE_DIR = os.getcwd()
PROM_BIN = f"{BASE_DIR}/prometheus_bin/prometheus"
GRAFANA_BIN = f"{BASE_DIR}/grafana_bin/bin/grafana-server"
NODE_EXP_BIN = f"{BASE_DIR}/prometheus_bin/node_exporter"

def start_process(cmd, log_file):
    with open(log_file, "w") as f:
        return subprocess.Popen(cmd.split(), stdout=f, stderr=subprocess.STDOUT)

# 1. Start Node Exporter
p_node = start_process(f"{NODE_EXP_BIN}", "node_exp.log")
print(f"🚀 Node Exporter started (PID: {p_node.pid})")

# 2. Start Prometheus
p_prom = start_process(f"{PROM_BIN} --config.file=prometheus.yml", "prometheus.log")
print(f"🚀 Prometheus started (PID: {p_prom.pid})")

# 3. Start Grafana
grafana_cmd = f"{GRAFANA_BIN} --homepath {BASE_DIR}/grafana_bin"
p_graf = start_process(grafana_cmd, "grafana.log")
print(f"🚀 Grafana started (PID: {p_graf.pid})")

## 4. Access Dashboard
Grafana is now running on **Port 3000**.

**1. Login:**
* Open the link below.
* User: `admin` / Password: `admin`.

**2. Add Data Source:**
* Go to **Connections** -> **Data Sources** -> **Add data source**.
* Select **Prometheus**.
* **IMPORTANT:** Set URL to `http://127.0.0.1:9090` (Do not use localhost).
* Click **Save & Test**.

**3. Import Dashboard:**
* Go to **Dashboards** -> **New** -> **Import**.
* Enter ID **1860** (Node Exporter Full) and click **Load**.
* Select your Prometheus source and click **Import**.

In [ ]:
print("📊 Access Grafana here:")
print("https://<YOUR_SATURN_CLOUD_URL>/proxy/3000/")
print("(Note: Replace <YOUR_SATURN_CLOUD_URL> with your browser URL domain, or localhost:3000 if local)")

In [ ]:
# 🛑 STOP ALL SERVICES
# Uncomment and run this cell to kill the background processes

p_node.terminate()
p_prom.terminate()
p_graf.terminate()
print("🛑 All services stopped.")

## 🏁 Conclusion

This template provides immediate visibility into your compute resources without complex installation steps. It helps you identify bottlenecks—such as single-core saturation or memory limits—during your data processing jobs.

To persist these metrics long-term or monitor multiple nodes, consider deploying this stack as a dedicated service on [Saturn Cloud](https://saturncloud.io/).